# OO vs OX Similarity Distribution Analysis (v2)
## Label-Aware Model-Dataset Fit Evaluation

### Changes from v1:
- **Timing**: Each model-dataset pair records computation time (seconds)
- **Pre-defined OO/OX pairs**: Pair indices computed once per dataset, reused across all models
- **Vectorized similarity**: Matrix cosine similarity instead of double for-loop
- **Dataset-centric architecture**: Load dataset once, iterate models → reduces HuggingFace API calls

### Metrics Calculated:
- Mean and Skewness for both OO and OX distributions
- Shannon Entropy for both distributions
- Chi-square between OO and OX distributions (measuring separation)
- Computation time per model-dataset pair

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import BertModel, BertTokenizer, AutoModel, AutoTokenizer
import torch
from scipy.stats import skew
from datasets import load_dataset
import warnings
import os
import time
from itertools import combinations
from datetime import datetime

warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA RTX A4000


## 1. Configuration

In [3]:
# ============================================================
# CONFIGURATION - Easy to modify
# ============================================================
# Sampling parameters
SAMPLE_SIZE = 300  # Total samples per dataset-model pair
NUM_STRATA = 150   # Number of strata (150 strata x 2 per stratum = 300)
N_BINS = 50        # Number of histogram bins

# Output folder
OUTPUT_DIR = 'oo_ox_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/histograms', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/csv', exist_ok=True)

# Models (7 models)
models_config = {
    'BERT': 'google-bert/bert-base-uncased',
    'SciBERT': 'allenai/scibert_scivocab_uncased',
    'LegalBERT': 'nlpaueb/legal-bert-base-uncased',
    'FinancialBERT': 'ahmedrachid/FinancialBERT',
    'PharmBERT': 'Lianglab/PharmBERT-uncased',
    'AgricultureBERT': 'recobo/agriculture-bert-uncased',
    'ChemicalBERT': 'recobo/chemical-bert-uncased',
    'BioBERT' : 'dmis-lab/biobert-base-cased-v1.1',
    'PubMedBERT' : 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract'
}

# Datasets (8 datasets - ALL using train split only)
datasets_config = {
    # ===== ORIGINAL DATASETS =====
    'Financial': {
        'source': 'huggingface',
        'path': 'takala/financial_phrasebank',
        'subset': 'sentences_50agree',
        'split': 'train',
        'text_column': 'sentence',
        'label_column': 'label'  # 3 classes: negative, neutral, positive
    },

    'ADE_Corpus': {
        'source': 'huggingface',
        'path': 'SetFit/ade_corpus_v2_classification',
        'subset': 'Ade_corpus_v2_classification',
        'split': 'train',
        'text_column': 'text',
        'label_column': 'label'  # Binary: ADE-related or not
    },

    'Climate_Detection': {
        'source': 'huggingface',
        'path': 'climatebert/climate_detection',
        'subset': None,
        'split': 'train',
        'text_column': 'text',
        'label_column': 'label'  # Binary: 0=not climate-related, 1=climate-related
    },

    'DrugReviews': {
        'source': 'huggingface',
        'path': 'forwins/Drug-Review-Dataset',
        'subset': None,
        'split': 'train',
        'text_column': 'review',
        'label_column': 'condition'  # Many classes (~900 medical conditions)
    },

    # ===== NEW DATASETS =====
    'IMDB': {
        'source': 'huggingface',
        'path': 'stanfordnlp/imdb',
        'subset': None,
        'split': 'train',  # ~25k samples
        'text_column': 'text',
        'label_column': 'label'  # Binary sentiment: 0=negative, 1=positive
    },

    'SST2': {
        'source': 'huggingface',
        'path': 'stanfordnlp/sst2',
        'subset': None,
        'split': 'train',  # ~67k samples
        'text_column': 'sentence',
        'label_column': 'label'  # Binary sentiment: 0=negative, 1=positive
    },

    'SciCite': {
        'source': 'huggingface',
        'path': 'allenai/scicite',
        'subset': None,
        'split': 'train',  # ~8k samples
        'text_column': 'string',  # Citation context string
        'label_column': 'label'   # 3 classes: 0=method, 1=background, 2=result
    },

    'PubMed_RCT': {
        'source': 'huggingface',
        'path': 'armanc/pubmed-rct20k',
        'subset': None,
        'split': 'train',  # ~15k samples
        'text_column': 'text',
        'label_column': 'label'  # 5 classes: background, objective, method, result, conclusion
    },
}

print("Configuration loaded!")
print(f"\nModels ({len(models_config)}): {list(models_config.keys())}")
print(f"Datasets ({len(datasets_config)}): {list(datasets_config.keys())}")
print(f"\nTotal model-dataset pairs: {len(models_config) * len(datasets_config)}")
print(f"Output directory: {OUTPUT_DIR}/")

Configuration loaded!

Models (9): ['BERT', 'SciBERT', 'LegalBERT', 'FinancialBERT', 'PharmBERT', 'AgricultureBERT', 'ChemicalBERT', 'BioBERT', 'PubMedBERT']
Datasets (8): ['Financial', 'ADE_Corpus', 'Climate_Detection', 'DrugReviews', 'IMDB', 'SST2', 'SciCite', 'PubMed_RCT']

Total model-dataset pairs: 72
Output directory: oo_ox_results/


## 2. Data Loading and Sampling Functions

In [4]:
def load_dataset_from_config(dataset_config):
    """
    Load dataset from HuggingFace
    Returns DataFrame with text and label columns
    """
    print(f"  Loading from HuggingFace: {dataset_config['path']}")

    if dataset_config['subset']:
        print(f"  Subset: {dataset_config['subset']}")
        dataset = load_dataset(
            dataset_config['path'],
            dataset_config['subset'],
            split=dataset_config['split']
        )
    else:
        dataset = load_dataset(
            dataset_config['path'],
            split=dataset_config['split']
        )

    df = dataset.to_pandas()

    text_col = dataset_config['text_column']
    label_col = dataset_config['label_column']

    if text_col not in df.columns:
        raise ValueError(f"Text column '{text_col}' not in dataset. Available: {df.columns.tolist()}")
    if label_col not in df.columns:
        raise ValueError(f"Label column '{label_col}' not in dataset. Available: {df.columns.tolist()}")

    print(f"  ✅ Loaded {len(df)} samples")
    print(f"  Text column: '{text_col}', Label column: '{label_col}'")

    return df, text_col, label_col


def clean_text(text):
    """Simple text cleaning"""
    import re
    text = str(text)
    text = re.sub(r'[?|!|\'|\"|#]', '', text)
    text = re.sub(r'[.|,|)|\(|\\\\|/]', ' ', text)
    text = text.strip().replace('\n', ' ')
    return text

In [5]:
def stratified_sample_with_labels(df, text_column, label_column, sample_size=300, num_strata=150):
    """
    Perform stratified sampling based on text length, preserving labels.

    Returns:
        List of tuples: [(text, label), (text, label), ...]
    """
    # Clean data
    df = df.dropna(subset=[text_column]).drop_duplicates(subset=[text_column])
    df = df[df[text_column].astype(str).str.len() > 10]

    if len(df) == 0:
        raise ValueError("No valid texts after cleaning")

    print(f"  Dataset size after cleaning: {len(df)} samples")

    # Calculate text lengths
    df = df.copy()
    df['length'] = df[text_column].apply(lambda x: len(str(x).split()))

    # Adaptive num_strata
    max_strata = max(10, len(df) // 5)
    num_strata = min(num_strata, max_strata)
    print(f"  Using {num_strata} strata")

    # If dataset is too small
    if len(df) < sample_size:
        print(f"  ⚠️ Dataset smaller than requested. Using all {len(df)} samples.")
        return list(zip(df[text_column].tolist(), df[label_column].tolist()))

    try:
        # Create quantile-based strata
        quantiles = df['length'].quantile([i/num_strata for i in range(1, num_strata)]).tolist()
        quantiles = np.unique(quantiles).tolist()

        # Filter quantiles too close together
        filtered_quantiles = []
        last_q = df['length'].min()
        for q in quantiles:
            if q - last_q > 1:
                filtered_quantiles.append(q)
                last_q = q

        if len(filtered_quantiles) == 0:
            print(f"  ⚠️ Text lengths too uniform. Using random sampling.")
            sampled = df.sample(n=sample_size, random_state=42)
            return list(zip(sampled[text_column].tolist(), sampled[label_column].tolist()))

        quantiles = filtered_quantiles
        bins = [df['length'].min() - 0.1] + quantiles + [df['length'].max() + 0.1]
        labels_list = [f'Q{i+1}' for i in range(len(bins) - 1)]

        df['strata'] = pd.cut(df['length'], bins=bins, labels=labels_list, include_lowest=True, duplicates='drop')
        df = df.dropna(subset=['strata'])

        if len(df) == 0:
            raise ValueError("All samples filtered out during stratification")

        # Calculate samples per stratum
        num_actual_strata = len(df['strata'].unique())
        n_per_stratum = sample_size // num_actual_strata

        strata_sizes = df.groupby('strata').size()
        min_samples = strata_sizes.min()

        if min_samples < n_per_stratum:
            n_per_stratum = min_samples
            print(f"  ⚠️ Adjusted to {n_per_stratum} per stratum (limited by smallest stratum)")

        if n_per_stratum < 1:
            print(f"  ⚠️ Too many strata. Using random sampling.")
            sampled = df.sample(n=min(sample_size, len(df)), random_state=42)
        else:
            print(f"  Sampling {n_per_stratum} texts from each of {num_actual_strata} strata...")
            sampled = df.groupby('strata', group_keys=False).apply(
                lambda x: x.sample(min(len(x), n_per_stratum), random_state=42)
            ).reset_index(drop=True)

        print(f"  ✅ Sampled {len(sampled)} texts")

        return list(zip(sampled[text_column].tolist(), sampled[label_column].tolist()))

    except Exception as e:
        print(f"  ⚠️ Stratification failed: {e}")
        print(f"  Falling back to random sampling...")
        sampled = df.sample(n=min(sample_size, len(df)), random_state=42)
        return list(zip(sampled[text_column].tolist(), sampled[label_column].tolist()))

## 3. Pre-compute OO/OX Pair Indices (NEW)

**Key optimization**: OO/OX pair assignment depends only on labels, not on the model.
We compute pair indices once per dataset and reuse them for all 7 models.

In [6]:
def labels_match(label1, label2):
    """
    Check if two labels match (handles multi-label case)

    For multi-label: returns True if they share ANY common label
    For single-label: returns True if labels are equal
    """
    if isinstance(label1, list) and isinstance(label2, list):
        return len(set(label1) & set(label2)) > 0
    return label1 == label2


def precompute_pair_indices(labels):
    """
    Pre-compute OO and OX pair indices from labels.
    
    This only depends on the labels (i.e., the dataset + sampling),
    NOT on the model. So we compute this ONCE per dataset and reuse
    across all models.
    
    Returns:
        oo_pairs: list of (i, j) tuples where labels[i] == labels[j]
        ox_pairs: list of (i, j) tuples where labels[i] != labels[j]
    """
    n = len(labels)
    oo_pairs = []
    ox_pairs = []
    
    for i in range(n):
        for j in range(i + 1, n):
            if labels_match(labels[i], labels[j]):
                oo_pairs.append((i, j))
            else:
                ox_pairs.append((i, j))
    
    return oo_pairs, ox_pairs


print("✅ Pair index functions defined!")

✅ Pair index functions defined!


## 4. Embedding and Vectorized Similarity Functions

In [7]:
def compute_embeddings(texts, model, tokenizer, max_length=512, batch_size=32):
    """
    Compute embeddings for a list of texts
    """
    model.eval()
    model = model.to(device)
    embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(
            batch_texts, 
            return_tensors="pt", 
            padding=True, 
            max_length=max_length, 
            truncation=True
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            batch_embeddings = outputs.last_hidden_state.mean(dim=1)
            embeddings.append(batch_embeddings.cpu())

    return torch.cat(embeddings, dim=0)


def compute_oo_ox_similarities_vectorized(embeddings, oo_pairs, ox_pairs):
    """
    Compute cosine similarities using vectorized matrix operations
    with pre-computed OO/OX pair indices.
    
    Instead of a double for-loop over all pairs, we:
    1. Normalize all embeddings once
    2. Compute the full cosine similarity matrix via matrix multiply
    3. Index into it using pre-computed OO and OX pair indices
    
    For n=300: matrix multiply computes 300x300=90,000 similarities at once
    vs. iterating through ~44,850 pairs individually.
    
    Args:
        embeddings: tensor of shape (n, d)
        oo_pairs: list of (i, j) index tuples for same-label pairs
        ox_pairs: list of (i, j) index tuples for different-label pairs
    
    Returns:
        oo_similarities: numpy array
        ox_similarities: numpy array
    """
    # Normalize embeddings
    embeddings_norm = torch.nn.functional.normalize(embeddings, p=2, dim=1)
    
    # Compute full cosine similarity matrix: (n, n)
    sim_matrix = torch.mm(embeddings_norm, embeddings_norm.t()).numpy()
    
    # Extract OO and OX similarities using pre-computed indices
    if len(oo_pairs) > 0:
        oo_i, oo_j = zip(*oo_pairs)
        oo_similarities = sim_matrix[np.array(oo_i), np.array(oo_j)]
    else:
        oo_similarities = np.array([])
    
    if len(ox_pairs) > 0:
        ox_i, ox_j = zip(*ox_pairs)
        ox_similarities = sim_matrix[np.array(ox_i), np.array(ox_j)]
    else:
        ox_similarities = np.array([])
    
    print(f"    OO pairs (same label): {len(oo_similarities):,}")
    print(f"    OX pairs (different label): {len(ox_similarities):,}")
    
    return oo_similarities, ox_similarities


print("✅ Embedding and vectorized similarity functions defined!")

✅ Embedding and vectorized similarity functions defined!


## 5. Statistical Functions

In [8]:
def calculate_entropy(similarities, n_bins=50):
    """
    Calculate Shannon entropy of similarity distribution using natural log.
    Skips bins with 0 counts.

    H = -Σ (p_i × ln(p_i))
    """
    if len(similarities) == 0:
        return np.nan

    counts, _ = np.histogram(similarities, bins=n_bins, 
                             range=(similarities.min(), similarities.max()))

    # Convert to probabilities, skip zeros
    probabilities = counts / counts.sum()
    probabilities = probabilities[probabilities > 0]

    # Shannon entropy with natural log
    entropy = -np.sum(probabilities * np.log(probabilities))

    return entropy


def calculate_chi_square_between_distributions(dist1, dist2, n_bins=50):
    """
    Calculate chi-square between two empirical distributions (OO vs OX).

    Uses dist1 (OO) as Expected and dist2 (OX) as Observed.
    Combines bins with low expected counts (< 5).

    χ² = Σ [(O_i - E_i)² / E_i]

    Returns:
        Normalized chi-square (χ² / degrees_of_freedom)
    """
    if len(dist1) == 0 or len(dist2) == 0:
        return np.nan

    # Use common range for both distributions
    combined = np.concatenate([dist1, dist2])
    bin_range = (combined.min(), combined.max())

    # Create histograms with same bins
    expected_counts, bin_edges = np.histogram(dist1, bins=n_bins, range=bin_range)
    observed_counts, _ = np.histogram(dist2, bins=n_bins, range=bin_range)

    # Normalize to same total (so we compare shapes, not sizes)
    total_observed = observed_counts.sum()
    total_expected = expected_counts.sum()

    if total_expected == 0:
        return np.nan

    expected_counts = expected_counts * (total_observed / total_expected)

    # Combine bins with low expected counts (< 5)
    combined_observed = []
    combined_expected = []
    current_obs = 0
    current_exp = 0

    for obs, exp in zip(observed_counts, expected_counts):
        current_obs += obs
        current_exp += exp

        if current_exp >= 5:
            combined_observed.append(current_obs)
            combined_expected.append(current_exp)
            current_obs = 0
            current_exp = 0

    # Add remaining to last bin if there's leftover
    if current_exp > 0:
        if len(combined_expected) > 0:
            combined_observed[-1] += current_obs
            combined_expected[-1] += current_exp
        else:
            combined_observed.append(current_obs)
            combined_expected.append(current_exp)

    combined_observed = np.array(combined_observed)
    combined_expected = np.array(combined_expected)

    # Need at least 2 bins
    if len(combined_expected) < 2:
        return np.nan

    # Filter out zero expected
    mask = combined_expected > 0
    combined_observed = combined_observed[mask]
    combined_expected = combined_expected[mask]

    if len(combined_expected) < 2:
        return np.nan

    # Calculate chi-square
    chi_square = np.sum((combined_observed - combined_expected)**2 / combined_expected)

    # Normalize by degrees of freedom
    degrees_of_freedom = len(combined_expected) - 1
    normalized_chi_square = chi_square / degrees_of_freedom if degrees_of_freedom > 0 else np.nan

    return normalized_chi_square


print("✅ Statistical functions defined!")

✅ Statistical functions defined!


## 6. Visualization Functions

In [9]:
def plot_oo_ox_histogram(oo_sims, ox_sims, model_name, dataset_name, stats, save_dir, n_bins=50):
    """
    Create overlaid histogram of OO vs OX distributions.
    Now includes computation time in subtitle.
    """
    fig, ax = plt.subplots(figsize=(12, 8))

    # Determine common range
    if len(oo_sims) > 0 and len(ox_sims) > 0:
        combined = np.concatenate([oo_sims, ox_sims])
        bin_range = (combined.min(), combined.max())
    elif len(oo_sims) > 0:
        bin_range = (oo_sims.min(), oo_sims.max())
    else:
        bin_range = (ox_sims.min(), ox_sims.max())

    # Plot OO histogram
    if len(oo_sims) > 0:
        ax.hist(oo_sims, bins=n_bins, range=bin_range, alpha=0.6, 
                label=f'OO (Same Label)\nn={len(oo_sims):,}\nμ={stats["oo_mean"]:.4f}\nskew={stats["oo_skewness"]:.4f}',
                color='blue', edgecolor='darkblue', linewidth=0.5, density=True)

    # Plot OX histogram
    if len(ox_sims) > 0:
        ax.hist(ox_sims, bins=n_bins, range=bin_range, alpha=0.6,
                label=f'OX (Different Label)\nn={len(ox_sims):,}\nμ={stats["ox_mean"]:.4f}\nskew={stats["ox_skewness"]:.4f}',
                color='red', edgecolor='darkred', linewidth=0.5, density=True)

    # Labels and title (now includes time)
    ax.set_xlabel('Cosine Similarity', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.set_title(f'{model_name} on {dataset_name}\n'
                 f'χ² (OO vs OX) = {stats["chi_square"]:.4f} | '
                 f'H(OO) = {stats["oo_entropy"]:.4f} | H(OX) = {stats["ox_entropy"]:.4f} | '
                 f'Time = {stats["time_seconds"]:.1f}s',
                 fontsize=14, fontweight='bold')

    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3)

    # Save
    filename = f"{save_dir}/{model_name}_{dataset_name}_oo_ox.png"
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()

    return filename


def create_chi_square_heatmap(all_results, save_dir):
    """
    Create heatmap of chi-square values across all model-dataset pairs.
    """
    model_names = list(all_results.keys())
    dataset_names = list(list(all_results.values())[0].keys())

    chi_matrix = np.zeros((len(model_names), len(dataset_names)))

    for i, model in enumerate(model_names):
        for j, dataset in enumerate(dataset_names):
            if dataset in all_results[model]:
                chi_matrix[i, j] = all_results[model][dataset]['chi_square']
            else:
                chi_matrix[i, j] = np.nan

    fig, ax = plt.subplots(figsize=(12, 8))
    im = ax.imshow(chi_matrix, aspect='auto', cmap='YlOrRd')

    ax.set_xticks(range(len(dataset_names)))
    ax.set_yticks(range(len(model_names)))
    ax.set_xticklabels(dataset_names, rotation=45, ha='right', fontsize=11)
    ax.set_yticklabels(model_names, fontsize=11)

    for i in range(len(model_names)):
        for j in range(len(dataset_names)):
            if not np.isnan(chi_matrix[i, j]):
                ax.text(j, i, f'{chi_matrix[i, j]:.2f}', 
                       ha='center', va='center', color='black', fontsize=10, fontweight='bold')

    ax.set_title('Chi-Square (OO vs OX)', fontsize=14, fontweight='bold')
    plt.colorbar(im, ax=ax, label='χ² (normalized)')
    plt.tight_layout()

    filename = f"{save_dir}/chi_square_heatmap.png"
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()

    print(f"  Heatmap saved to: {filename}")
    return filename


print("✅ Visualization functions defined!")

✅ Visualization functions defined!


## 7. Run Full Analysis (Restructured: Dataset-Centric)

### Architecture Change:
**Before (v1):** For each model → for each dataset → load dataset, sample, compute pairs, compute embeddings  
**After (v2):** For each dataset → load once, sample once, pre-compute pairs → for each model → load model, compute embeddings, look up similarities

### Benefits:
- Dataset loaded **1x** instead of **7x** (once per dataset vs once per model-dataset pair)
- OO/OX pair indices computed **1x per dataset** (guaranteed identical across models)
- Reduces HuggingFace API calls from 56 dataset loads to 8
- Vectorized similarity computation via matrix multiply

In [10]:
# Store all results
all_results = {model: {} for model in models_config.keys()}

# Store timing info separately for easy access
timing_results = []

# Track progress
total_pairs = len(models_config) * len(datasets_config)
current_pair = 0

print(f"🚀 Starting OO vs OX Analysis (v2 - Dataset-Centric)")
print(f"   Models: {len(models_config)}")
print(f"   Datasets: {len(datasets_config)}")
print(f"   Total pairs: {total_pairs}")
print(f"   Sample size: {SAMPLE_SIZE}")
print(f"   Output: {OUTPUT_DIR}/")
print("\n" + "="*80)

overall_start_time = datetime.now()

# ============================================================
# OUTER LOOP: Iterate over DATASETS (load once, reuse for all models)
# ============================================================
for dataset_name, dataset_config in datasets_config.items():
    print(f"\n{'#'*80}")
    print(f"# DATASET: {dataset_name}")
    print(f"{'#'*80}")
    
    # --- Step 1: Load dataset ONCE ---
    try:
        ds_load_start = time.time()
        df, text_col, label_col = load_dataset_from_config(dataset_config)
        ds_load_time = time.time() - ds_load_start
        print(f"  📦 Dataset loaded in {ds_load_time:.1f}s")
    except Exception as e:
        print(f"  ❌ Failed to load dataset {dataset_name}: {e}")
        import traceback
        traceback.print_exc()
        continue
    
    # --- Step 2: Sample ONCE ---
    sample_start = time.time()
    samples = stratified_sample_with_labels(df, text_col, label_col, SAMPLE_SIZE, NUM_STRATA)
    texts = [clean_text(s[0]) for s in samples]
    labels = [s[1] for s in samples]
    sample_time = time.time() - sample_start
    
    # Check label distribution
    unique_labels = set()
    for label in labels:
        if isinstance(label, list):
            unique_labels.update(label)
        else:
            unique_labels.add(label)
    print(f"  Unique labels in sample: {len(unique_labels)}")
    print(f"  📋 Sampling done in {sample_time:.1f}s")
    
    # --- Step 3: Pre-compute OO/OX pair indices ONCE ---
    pair_start = time.time()
    oo_pairs, ox_pairs = precompute_pair_indices(labels)
    pair_time = time.time() - pair_start
    
    n_total_pairs = len(oo_pairs) + len(ox_pairs)
    print(f"  🔗 Pre-computed pair indices in {pair_time:.2f}s")
    print(f"     OO pairs: {len(oo_pairs):,} | OX pairs: {len(ox_pairs):,} | Total: {n_total_pairs:,}")
    print(f"     (These counts are FIXED for all models on {dataset_name})")
    
    # Free the DataFrame to save memory
    del df
    
    # ============================================================
    # INNER LOOP: Iterate over MODELS (only load model + compute embeddings)
    # ============================================================
    for model_name, model_path in models_config.items():
        current_pair += 1
        print(f"\n  [{current_pair}/{total_pairs}] {model_name} on {dataset_name}")
        print(f"  {'-'*60}")
        
        pair_start_time = time.time()
        
        try:
            # --- Load model ---
            print(f"    Loading model: {model_path}")
            model_load_start = time.time()
            try:
                tokenizer = BertTokenizer.from_pretrained(model_path)
                model = BertModel.from_pretrained(model_path)
            except:
                tokenizer = AutoTokenizer.from_pretrained(model_path)
                model = AutoModel.from_pretrained(model_path)
            model_load_time = time.time() - model_load_start
            print(f"    ✅ Model loaded in {model_load_time:.1f}s")
            
            # --- Compute embeddings ---
            print(f"    Computing embeddings for {len(texts)} texts...")
            embed_start = time.time()
            embeddings = compute_embeddings(texts, model, tokenizer)
            embed_time = time.time() - embed_start
            print(f"    ✅ Embeddings computed in {embed_time:.1f}s")
            
            # --- Compute similarities using pre-computed pair indices (VECTORIZED) ---
            print(f"    Computing OO/OX similarities (vectorized)...")
            sim_start = time.time()
            oo_sims, ox_sims = compute_oo_ox_similarities_vectorized(
                embeddings, oo_pairs, ox_pairs
            )
            sim_time = time.time() - sim_start
            print(f"    ✅ Similarities computed in {sim_time:.2f}s")
            
            # --- Calculate statistics ---
            total_pair_time = time.time() - pair_start_time
            
            results = {
                'oo_similarities': oo_sims,
                'ox_similarities': ox_sims,
                'n_oo_pairs': len(oo_sims),
                'n_ox_pairs': len(ox_sims),
                
                # OO statistics
                'oo_mean': np.mean(oo_sims) if len(oo_sims) > 0 else np.nan,
                'oo_skewness': skew(oo_sims, bias=False) if len(oo_sims) > 2 else np.nan,
                'oo_entropy': calculate_entropy(oo_sims, N_BINS) if len(oo_sims) > 0 else np.nan,
                
                # OX statistics
                'ox_mean': np.mean(ox_sims) if len(ox_sims) > 0 else np.nan,
                'ox_skewness': skew(ox_sims, bias=False) if len(ox_sims) > 2 else np.nan,
                'ox_entropy': calculate_entropy(ox_sims, N_BINS) if len(ox_sims) > 0 else np.nan,
                
                # Chi-square between distributions
                'chi_square': calculate_chi_square_between_distributions(oo_sims, ox_sims, N_BINS),
                
                # Timing (Feature 1)
                'time_seconds': total_pair_time,
            }
            
            all_results[model_name][dataset_name] = results
            
            # Store timing detail
            timing_results.append({
                'Model': model_name,
                'Dataset': dataset_name,
                'Total_Time_s': round(total_pair_time, 2),
                'Model_Load_s': round(model_load_time, 2),
                'Embedding_s': round(embed_time, 2),
                'Similarity_s': round(sim_time, 2),
            })
            
            # Print results
            print(f"\n    Results:")
            print(f"      OO: mean={results['oo_mean']:.4f}, skew={results['oo_skewness']:.4f}, H={results['oo_entropy']:.4f}")
            print(f"      OX: mean={results['ox_mean']:.4f}, skew={results['ox_skewness']:.4f}, H={results['ox_entropy']:.4f}")
            print(f"      χ² (OO vs OX): {results['chi_square']:.4f}")
            print(f"      ⏱️  Time: {total_pair_time:.1f}s (model:{model_load_time:.1f}s + embed:{embed_time:.1f}s + sim:{sim_time:.2f}s)")
            
            # Create histogram
            plot_oo_ox_histogram(
                results['oo_similarities'],
                results['ox_similarities'],
                model_name, dataset_name,
                results,
                f"{OUTPUT_DIR}/histograms",
                n_bins=N_BINS
            )
            print(f"    ✅ Histogram saved")
            
            # Clean up GPU memory
            del model, embeddings
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
        except Exception as e:
            print(f"    ❌ Error: {e}")
            import traceback
            traceback.print_exc()
            
            # Record failed timing
            timing_results.append({
                'Model': model_name,
                'Dataset': dataset_name,
                'Total_Time_s': round(time.time() - pair_start_time, 2),
                'Model_Load_s': np.nan,
                'Embedding_s': np.nan,
                'Similarity_s': np.nan,
            })

overall_elapsed = datetime.now() - overall_start_time
print(f"\n\n{'='*80}")
print(f"✅ Analysis complete!")
print(f"   Time elapsed: {overall_elapsed}")
print(f"   Results saved to: {OUTPUT_DIR}/")

🚀 Starting OO vs OX Analysis (v2 - Dataset-Centric)
   Models: 9
   Datasets: 8
   Total pairs: 72
   Sample size: 300
   Output: oo_ox_results/


################################################################################
# DATASET: Financial
################################################################################
  Loading from HuggingFace: takala/financial_phrasebank
  Subset: sentences_50agree
  ✅ Loaded 4846 samples
  Text column: 'sentence', Label column: 'label'
  📦 Dataset loaded in 0.6s
  Dataset size after cleaning: 4837 samples
  Using 150 strata
  Sampling 12 texts from each of 24 strata...
  ✅ Sampled 288 texts
  Unique labels in sample: 3
  📋 Sampling done in 0.0s
  🔗 Pre-computed pair indices in 0.01s
     OO pairs: 18,867 | OX pairs: 22,461 | Total: 41,328
     (These counts are FIXED for all models on Financial)

  [1/72] BERT on Financial
  ------------------------------------------------------------
    Loading model: google-bert/bert-base-uncased


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

    ✅ Model loaded in 1.4s
    Computing embeddings for 297 texts...
    ✅ Embeddings computed in 0.7s
    Computing OO/OX similarities (vectorized)...
    OO pairs (same label): 12,837
    OX pairs (different label): 31,119
    ✅ Similarities computed in 0.31s

    Results:
      OO: mean=0.7383, skew=-0.5828, H=3.4535
      OX: mean=0.6955, skew=-0.3959, H=3.4128
      χ² (OO vs OX): 160.8382
      ⏱️  Time: 2.4s (model:1.4s + embed:0.7s + sim:0.31s)
    ✅ Histogram saved

  [69/72] AgricultureBERT on PubMed_RCT
  ------------------------------------------------------------
    Loading model: recobo/agriculture-bert-uncased


Some weights of BertModel were not initialized from the model checkpoint at recobo/agriculture-bert-uncased and are newly initialized: ['bert.pooler.dense.weight', 'bert.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    ✅ Model loaded in 0.8s
    Computing embeddings for 297 texts...
    ✅ Embeddings computed in 0.6s
    Computing OO/OX similarities (vectorized)...
    OO pairs (same label): 12,837
    OX pairs (different label): 31,119
    ✅ Similarities computed in 0.01s

    Results:
      OO: mean=0.8181, skew=-0.5232, H=3.3426
      OX: mean=0.7976, skew=-0.5617, H=3.0797
      χ² (OO vs OX): 173.8195
      ⏱️  Time: 1.4s (model:0.8s + embed:0.6s + sim:0.01s)
    ✅ Histogram saved

  [70/72] ChemicalBERT on PubMed_RCT
  ------------------------------------------------------------
    Loading model: recobo/chemical-bert-uncased


Some weights of BertModel were not initialized from the model checkpoint at recobo/chemical-bert-uncased and are newly initialized: ['bert.pooler.dense.weight', 'bert.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    ✅ Model loaded in 0.8s
    Computing embeddings for 297 texts...
    ✅ Embeddings computed in 0.6s
    Computing OO/OX similarities (vectorized)...
    OO pairs (same label): 12,837
    OX pairs (different label): 31,119
    ✅ Similarities computed in 0.01s

    Results:
      OO: mean=0.8249, skew=-1.7476, H=3.1690
      OX: mean=0.8083, skew=-1.6189, H=3.0124
      χ² (OO vs OX): 145.8606
      ⏱️  Time: 1.4s (model:0.8s + embed:0.6s + sim:0.01s)
    ✅ Histogram saved

  [71/72] BioBERT on PubMed_RCT
  ------------------------------------------------------------
    Loading model: dmis-lab/biobert-base-cased-v1.1
    ✅ Model loaded in 0.9s
    Computing embeddings for 297 texts...
    ✅ Embeddings computed in 0.7s
    Computing OO/OX similarities (vectorized)...
    OO pairs (same label): 12,837
    OX pairs (different label): 31,119
    ✅ Similarities computed in 0.03s

    Results:
      OO: mean=0.8765, skew=-0.6739, H=3.2941
      OX: mean=0.8655, skew=-0.6846, H=3.1782
     

## 8. Create Summary CSV (with Timing)

In [11]:
# Create summary DataFrame
summary_data = []

for model_name, datasets in all_results.items():
    for dataset_name, results in datasets.items():
        summary_data.append({
            'Model': model_name,
            'Dataset': dataset_name,
            'OO_Pairs': results['n_oo_pairs'],
            'OX_Pairs': results['n_ox_pairs'],
            'OO_Mean': results['oo_mean'],
            'OO_Skewness': results['oo_skewness'],
            'OO_Entropy': results['oo_entropy'],
            'OX_Mean': results['ox_mean'],
            'OX_Skewness': results['ox_skewness'],
            'OX_Entropy': results['ox_entropy'],
            'Chi_Square': results['chi_square'],
            'Mean_Difference': results['oo_mean'] - results['ox_mean'],
            'Time_Seconds': results['time_seconds'],
        })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.round(4)

# Save to CSV
csv_filename = f"{OUTPUT_DIR}/csv/oo_ox_summary.csv"
summary_df.to_csv(csv_filename, index=False)
print(f"✅ Summary saved to: {csv_filename}")

# Display
print("\n" + "="*80)
print("SUMMARY TABLE")
print("="*80)
display(summary_df)

✅ Summary saved to: oo_ox_results/csv/oo_ox_summary.csv

SUMMARY TABLE


,Model,Dataset,OO_Pairs,OX_Pairs,OO_Mean,OO_Skewness,OO_Entropy,OX_Mean,OX_Skewness,OX_Entropy,Chi_Square,Mean_Difference,Time_Seconds
0,BERT,Financial,18867,22461,0.6730,-0.4017,3.3424,0.6886,-0.3916,3.4133,14.6801,-0.0156,3.7186
1,BERT,ADE_Corpus,24402,16353,0.7345,-0.8353,3.2404,0.7441,-1.1079,3.1922,9.2106,-0.0096,1.4115
2,BERT,Climate_Detection,18731,10915,0.8086,-0.4658,3.1579,0.7867,-0.4782,3.0889,62.9559,0.0219,1.8723
3,BERT,DrugReviews,1480,30146,0.8751,-0.8822,3.3547,0.8582,-0.6384,3.3317,195.7586,0.0168,2.6279
4,BERT,IMDB,15576,15052,0.8681,-1.3690,3.2183,0.8664,-1.5962,3.0855,10.4076,0.0017,3.9854
...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,PubMedBERT,DrugReviews,1480,30146,0.9876,-1.8155,3.0380,0.9868,-1.9103,2.8566,168.0550,0.0009,2.3203
68,PubMedBERT,IMDB,15576,15052,0.9925,-1.9280,2.8626,0.9925,-2.0538,2.8159,3.8223,0.0000,4.0896
69,PubMedBERT,SST2,20673,20655,0.9794,-1.6398,2.7639,0.9792,-1.6435,2.8183,2.2534,0.0001,1.5093
70,PubMedBERT,SciCite,15907,22874,0.9712,-2.2547,2.6558,0.9705,-2.1714,2.6513,6.9890,0.0007,2.4341


## 9. Timing Summary

In [12]:
# Create detailed timing DataFrame
timing_df = pd.DataFrame(timing_results)

# Save timing CSV
timing_csv = f"{OUTPUT_DIR}/csv/timing_summary.csv"
timing_df.to_csv(timing_csv, index=False)
print(f"✅ Timing summary saved to: {timing_csv}")

# Display timing table
print("\n" + "="*80)
print("TIMING SUMMARY")
print("="*80)
display(timing_df)

# Timing statistics
print(f"\n📊 Timing Statistics:")
print(f"   Mean time per pair:   {timing_df['Total_Time_s'].mean():.1f}s")
print(f"   Min time per pair:    {timing_df['Total_Time_s'].min():.1f}s")
print(f"   Max time per pair:    {timing_df['Total_Time_s'].max():.1f}s")
print(f"   Total time:           {timing_df['Total_Time_s'].sum():.1f}s ({timing_df['Total_Time_s'].sum()/60:.1f}min)")

# Time breakdown
if timing_df['Model_Load_s'].notna().any():
    print(f"\n   ⏱️  Time Breakdown (averages):")
    print(f"      Model loading:     {timing_df['Model_Load_s'].mean():.1f}s")
    print(f"      Embedding:         {timing_df['Embedding_s'].mean():.1f}s")
    print(f"      Similarity:        {timing_df['Similarity_s'].mean():.2f}s")

✅ Timing summary saved to: oo_ox_results/csv/timing_summary.csv

TIMING SUMMARY


,Model,Dataset,Total_Time_s,Model_Load_s,Embedding_s,Similarity_s
0,BERT,Financial,3.72,3.06,0.63,0.03
1,SciBERT,Financial,3.85,3.36,0.46,0.02
2,LegalBERT,Financial,3.49,2.99,0.48,0.02
3,FinancialBERT,Financial,3.00,2.53,0.46,0.01
4,PharmBERT,Financial,3.18,2.58,0.45,0.15
...,...,...,...,...,...,...
67,PharmBERT,PubMed_RCT,2.38,1.41,0.66,0.31
68,AgricultureBERT,PubMed_RCT,1.42,0.84,0.57,0.01
69,ChemicalBERT,PubMed_RCT,1.38,0.81,0.57,0.01
70,BioBERT,PubMed_RCT,1.64,0.93,0.68,0.03



📊 Timing Statistics:
   Mean time per pair:   2.3s
   Min time per pair:    1.1s
   Max time per pair:    4.5s
   Total time:           166.2s (2.8min)

   ⏱️  Time Breakdown (averages):
      Model loading:     1.2s
      Embedding:         1.1s
      Similarity:        0.06s


## 10. Create Heatmaps

In [13]:
# Create chi-square heatmap
print("Creating chi-square heatmap...")
create_chi_square_heatmap(all_results, OUTPUT_DIR)

# Create mean difference heatmap (OO_mean - OX_mean)
print("\nCreating mean difference heatmap...")

model_names = list(all_results.keys())
dataset_names = list(list(all_results.values())[0].keys())

diff_matrix = np.zeros((len(model_names), len(dataset_names)))

for i, model in enumerate(model_names):
    for j, dataset in enumerate(dataset_names):
        if dataset in all_results[model]:
            r = all_results[model][dataset]
            diff_matrix[i, j] = r['oo_mean'] - r['ox_mean']
        else:
            diff_matrix[i, j] = np.nan

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(diff_matrix, aspect='auto', cmap='RdYlGn', vmin=-0.1, vmax=0.1)

ax.set_xticks(range(len(dataset_names)))
ax.set_yticks(range(len(model_names)))
ax.set_xticklabels(dataset_names, rotation=45, ha='right', fontsize=11)
ax.set_yticklabels(model_names, fontsize=11)

for i in range(len(model_names)):
    for j in range(len(dataset_names)):
        if not np.isnan(diff_matrix[i, j]):
            ax.text(j, i, f'{diff_matrix[i, j]:.4f}', 
                   ha='center', va='center', color='black', fontsize=10, fontweight='bold')

ax.set_title('Mean Difference (OO - OX) - Higher = Better Class Separation', 
             fontsize=14, fontweight='bold')

plt.colorbar(im, ax=ax, label='μ(OO) - μ(OX)')
plt.tight_layout()

filename = f"{OUTPUT_DIR}/mean_difference_heatmap.png"
plt.savefig(filename, dpi=150, bbox_inches='tight')
plt.close()

print(f"  Mean difference heatmap saved to: {filename}")

Creating chi-square heatmap...
  Heatmap saved to: oo_ox_results/chi_square_heatmap.png

Creating mean difference heatmap...
  Mean difference heatmap saved to: oo_ox_results/mean_difference_heatmap.png


## 11. Timing Heatmap

In [14]:
# Create timing heatmap
print("Creating timing heatmap...")

time_matrix = np.zeros((len(model_names), len(dataset_names)))

for i, model in enumerate(model_names):
    for j, dataset in enumerate(dataset_names):
        if dataset in all_results[model]:
            time_matrix[i, j] = all_results[model][dataset]['time_seconds']
        else:
            time_matrix[i, j] = np.nan

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(time_matrix, aspect='auto', cmap='YlOrBr')

ax.set_xticks(range(len(dataset_names)))
ax.set_yticks(range(len(model_names)))
ax.set_xticklabels(dataset_names, rotation=45, ha='right', fontsize=11)
ax.set_yticklabels(model_names, fontsize=11)

for i in range(len(model_names)):
    for j in range(len(dataset_names)):
        if not np.isnan(time_matrix[i, j]):
            ax.text(j, i, f'{time_matrix[i, j]:.1f}s', 
                   ha='center', va='center', color='black', fontsize=9, fontweight='bold')

ax.set_title('Computation Time per Model-Dataset Pair (seconds)', 
             fontsize=14, fontweight='bold')

plt.colorbar(im, ax=ax, label='Time (seconds)')
plt.tight_layout()

filename = f"{OUTPUT_DIR}/timing_heatmap.png"
plt.savefig(filename, dpi=150, bbox_inches='tight')
plt.close()

print(f"  Timing heatmap saved to: {filename}")

Creating timing heatmap...
  Timing heatmap saved to: oo_ox_results/timing_heatmap.png


## 12. Final Summary

In [15]:
print("\n" + "="*80)
print("📊 ANALYSIS COMPLETE (v2)")
print("="*80)

print(f"\n✅ Models analyzed: {len(models_config)}")
print(f"✅ Datasets analyzed: {len(datasets_config)}")
print(f"✅ Total pairs: {len(models_config) * len(datasets_config)}")

print(f"\n📁 Output files:")
print(f"   Histograms:      {OUTPUT_DIR}/histograms/")
print(f"   Summary CSV:     {OUTPUT_DIR}/csv/oo_ox_summary.csv")
print(f"   Timing CSV:      {OUTPUT_DIR}/csv/timing_summary.csv")
print(f"   Heatmaps:        {OUTPUT_DIR}/")

print(f"\n📈 Metrics calculated for each pair:")
print(f"   • OO: Mean, Skewness, Entropy")
print(f"   • OX: Mean, Skewness, Entropy")
print(f"   • Chi-square (OO vs OX)")
print(f"   • Mean difference (OO - OX)")
print(f"   • Computation time (seconds)")

print(f"\n🔍 Interpretation:")
print(f"   • Higher Chi-square = Better separation between same/different label pairs")
print(f"   • Positive Mean Difference = Same-label pairs are more similar (expected)")
print(f"   • OO should have higher mean than OX for a good model fit")

print(f"\n⚡ v2 Optimizations:")
print(f"   • Dataset loaded 1x per dataset (not 7x)")
print(f"   • OO/OX pair indices pre-computed once per dataset")
print(f"   • Vectorized cosine similarity via matrix multiply")
print(f"   • Per-pair timing recorded")


📊 ANALYSIS COMPLETE (v2)

✅ Models analyzed: 9
✅ Datasets analyzed: 8
✅ Total pairs: 72

📁 Output files:
   Histograms:      oo_ox_results/histograms/
   Summary CSV:     oo_ox_results/csv/oo_ox_summary.csv
   Timing CSV:      oo_ox_results/csv/timing_summary.csv
   Heatmaps:        oo_ox_results/

📈 Metrics calculated for each pair:
   • OO: Mean, Skewness, Entropy
   • OX: Mean, Skewness, Entropy
   • Chi-square (OO vs OX)
   • Mean difference (OO - OX)
   • Computation time (seconds)

🔍 Interpretation:
   • Higher Chi-square = Better separation between same/different label pairs
   • Positive Mean Difference = Same-label pairs are more similar (expected)
   • OO should have higher mean than OX for a good model fit

⚡ v2 Optimizations:
   • Dataset loaded 1x per dataset (not 7x)
   • OO/OX pair indices pre-computed once per dataset
   • Vectorized cosine similarity via matrix multiply
   • Per-pair timing recorded
